In [ ]:
# %%capture
# !pip install transformers qdrant_client
# !pip install --no-cache-dir langchain-community pypdf
# !pip install vllm

# Baseline:

## 1. Environment setup

In [2]:
import re
import uuid
import time
import torch
from tqdm import tqdm

from vllm import LLM, SamplingParams
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from langchain_community.document_loaders import PyPDFLoader
from qdrant_client import QdrantClient, models

/tmp/ipykernel_3130/2540285123.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 2. Data loading

Загрузка документов, очистка текста от лишних символов, пробелов и переносов '-\n'

In [ ]:
loader = PyPDFLoader(".../gost-r-59921.pdf")
documents = loader.load()

def clean_text(text):
    text = re.sub(r"[\u2002\u2003\u2009\u00a0]", " ", text)
    text = text.replace("-\n", "")
    text = re.sub(r"[^\S\n]+", " ", text)
    return text

## 3. Embedding model initialization & document chunking

Инициализация модели BAAI/bge-m3 для эмбеддингов

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
embedding_model = SentenceTransformer("BAAI/bge-m3", device=device, model_kwargs={"torch_dtype": torch.float16})

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="clinical_medicine",
    vectors_config=models.VectorParams(size=1024, distance=models.Distance.COSINE),
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

True

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n"],
    chunk_size=500,
    chunk_overlap=50
)

In [7]:
for doc in tqdm(documents):
    chunks = text_splitter.split_text(doc.page_content)
    chunks = [re.sub(r"\s+", " ", c).strip() for c in chunks]
    vectors = embedding_model.encode(chunks, normalize_embeddings=True, device=device).tolist()

    client.upsert(
        collection_name="clinical_medicine",
        points=[
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=vectors[j],
                payload={
                    "text": chunks[j],
                    "page": doc.metadata.get("page", 0),
                    "source": doc.metadata.get("source", ""),
                }
            )
            for j in range(len(chunks))
        ]
    )


100%|██████████| 32/32 [00:02<00:00, 12.75it/s]


## 4. LLM initialization

Инициализация генеративной модели

In [9]:
llm = LLM(
    model="Qwen/Qwen3-1.7B",
    dtype="float16",
    max_model_len=4096,
    gpu_memory_utilization=0.75,
    disable_log_stats=False
)

INFO 06-17 10:16:32 [utils.py:261] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.75, 'model': 'Qwen/Qwen3-1.7B'}


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

INFO 06-17 10:17:03 [model.py:541] Resolved architecture: Qwen3ForCausalLM
WARNING 06-17 10:17:03 [model.py:1885] Casting torch.bfloat16 to torch.float16.
INFO 06-17 10:17:03 [model.py:1561] Using max model len 4096
INFO 06-17 10:17:03 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-17 10:17:03 [vllm.py:624] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

WARNING 06-17 10:17:07 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 06-17 10:23:24 [llm.py:343] Supported tasks: ['generate']


In [10]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

In [11]:
!nvidia-smi

Wed Jun 17 10:23:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             29W /   70W |   13915MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 5. Retrieval & prompt

In [12]:
def retrieve(query, top_k=3):
    query_vector = embedding_model.encode(query, normalize_embeddings=True)
    results = client.query_points(
        collection_name="clinical_medicine",
        query=query_vector.tolist(),
        limit=top_k,
    )
    return results.points

In [13]:
def build_prompt(query, points):
    context = "\n\n".join(p.payload["text"] for p in points)
    messages = [
        {
            "role": "system",
            "content": "Ты - эксперт по нормативным документам в сфере ИИ. Отвечай только на основе предоставленного контекста. Если ответа в контексте нет - скажи об этом."
        },
        {
            "role": "user",
            "content": f"Контекст:\n{context}\n\nВопрос: {query}"
        }
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

## 6. Inference metrics

In [14]:
def generate_with_metrics(prompt, max_new_tokens=512):
    sampling_params = SamplingParams(max_tokens=max_new_tokens, temperature=0.0)

    gen_start = time.perf_counter()
    outputs = llm.generate(prompt, sampling_params)
    gen_end = time.perf_counter()

    output = outputs[0]
    n_tokens = len(output.outputs[0].token_ids)
    total_time = gen_end - gen_start

    metrics = output.metrics

    ttft = metrics.first_token_latency if metrics.first_token_latency else total_time / n_tokens
    itl = (metrics.last_token_ts - metrics.first_token_ts) / max(n_tokens - 1, 1) if metrics.first_token_ts else 0
    tps = n_tokens / total_time if total_time > 0 else 0

    return {
        "response": output.outputs[0].text,
        "metrics": {
            "n_tokens": n_tokens,
            "total_time": round(total_time, 3),
            "ttft": round(ttft, 3),
            "itl": round(itl, 3),
            "tps": round(tps, 2),
        }
    }

## 7. Baseline evaluation



*   Time To First Token (TTFT) на уровне ниже 1 секунды: префилл выполняется быстро, удачный размер контекста
*   Inter Token Latency (ITL) показывает быструю генерацию
*   Tokens Per Second (TPS)






In [15]:
def ask(query):
    points = retrieve(query)
    result = generate_with_metrics(build_prompt(query, points))
    return result

In [16]:
queries = [
    "Что должно быть оценено на первом этапе проведения испытаний СИИ?",
    "Какие методы могут быть использованы для проверки устойчивости СИИ к нарушениям СОП, артефактам данных и помехам?",
    "Какие требования предъявляются к точности воспроизведения сигналов в СИИ при испытаниях?",
    "По каким правилам составляется отчет о тестировании систем искусственного интеллекта для клинической физиологии?",
    "Какие метрики оценки должна определить СИИ при обработке сигнала?"
]

results = []
for query in tqdm(queries):
    result = ask(query)
    result["query"] = query
    results.append(result)
    print(f"\nQ: {query}")
    print(f"A: {result['response']}")
    print(f"Метрики: {result['metrics']}")

  0%|          | 0/5 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

 20%|██        | 1/5 [00:01<00:06,  1.61s/it]


Q: Что должно быть оценено на первом этапе проведения испытаний СИИ?
A: На первом этапе проведения испытаний СИИ должно быть оценено точность воспроизведения системы, то есть проверена её работоспособность и способность к выполнению заданных задач в условиях, аналогичных реальным.
Метрики: {'n_tokens': 56, 'total_time': 1.178, 'ttft': 0.161, 'itl': 0.018, 'tps': 47.55}


...


Q: По каким правилам составляется отчет о тестировании систем искусственного интеллекта для клинической физиологии?
A: Отчет о тестировании систем искусственного интеллекта для клинической физиологии составляется по правилам, определенным в **ГОСТ Р ИСО 14155**.
Метрики: {'n_tokens': 51, 'total_time': 0.981, 'ttft': 0.077, 'itl': 0.017, 'tps': 51.99}


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

100%|██████████| 5/5 [00:15<00:00,  3.04s/it]


Q: Какие метрики оценки должна определить СИИ при обработке сигнала?
A: СИИ должна определить сигнал как полученный от биологического объекта, в ходе подобного вида тестирований рекомендуется оценивать точность (precision), полноту (recall), f-оценку (f-score) и другие параметры.
Метрики: {'n_tokens': 67, 'total_time': 1.306, 'ttft': 0.087, 'itl': 0.018, 'tps': 51.31}


In [17]:
def summarize_metrics(results):
    metrics_keys = ["n_tokens", "total_time", "ttft", "itl", "tps"]

    summary = {}
    for key in metrics_keys:
        values = [r["metrics"][key] for r in results]
        summary[key] = {
            "mean": round(sum(values) / len(values), 3),
            "min":  round(min(values), 3),
            "max":  round(max(values), 3),
        }

    print(f"{'Метрика':<12} {'Mean':>8} {'Min':>8} {'Max':>8}")
    print("-" * 40)
    for key, stats in summary.items():
        print(f"{key:<12} {stats['mean']:>8} {stats['min']:>8} {stats['max']:>8}")

    return summary

summary = summarize_metrics(results)

Метрика          Mean      Min      Max
----------------------------------------
n_tokens        159.2       51      439
total_time      2.887    0.981    7.759
ttft            0.107    0.077    0.161
itl             0.017    0.017    0.018
tps            52.888    47.55    57.01
